In [ ]:
# importing Libraries
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from IPython.display import display, Markdown
from eli5.permutation_importance import get_score_importances
from tensorflow.keras import Sequential, layers, optimizers, losses

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, KBinsDiscretizer
# from sklearn.datasets import make_classification

In [ ]:
def printh(h="h1", text=""):
    display(Markdown('<{}>{}</{}>'.format(h, text, h)))

def show_hist(df, features, figwidth=12):
    len_ft = len(features)
    fig, axes = plt.subplots(1, len_ft, figsize=(figwidth,3))
    if isinstance(axes, np.ndarray) is False:
        axes = np.array([axes])
    for idx, ft in enumerate(features):
        sns.histplot(df[ft], kde=True, ax=axes[idx])
        axes[idx].set_title(ft)
    plt.tight_layout()
    plt.show()

def show_hists(df, features, n=4):
    loop = math.ceil(len(features) / n)
    for idx in range(loop):
        if idx == 0:
            show_hist(df=df, features=features[:n], figwidth=12)
        elif idx == (loop-1):
            remainder = len(features) - (idx * n)
            remainder_width = math.ceil(12 * remainder / n)
            show_hist(df=df, features=features[n*(loop-1):], figwidth=remainder_width)
        else:
            show_hist(df=df, features=features[n*idx:n*(idx+1)], figwidth=12)

def show_skew_kurt(df, print_result=True):
    numerical_columns = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
    skewness = df[numerical_columns].skew()
    kurtosis = df[numerical_columns].kurt()
    sk_df = pd.DataFrame(
        {"skewness": skewness, "kurtosis": kurtosis},
        index=numerical_columns,
    )
    if print_result:
        print(sk_df)
    return sk_df

def show_corr(df):
    corr = df.corr()
    size = corr.shape[0]
    size = min(size, 15)
    plt.figure(figsize=(size, size))
    sns.heatmap(df.corr(),annot=True,cbar=True,cmap='coolwarm')
    plt.show()

In [ ]:
printh("h1", "Load DataFrame")
main_df = pd.read_csv('noisy_xor2.csv')

printh("h2", "Main DF Info")
print(main_df.info())

printh("h2", "Is NaN")
print("main_df.isna().sum()")
print(main_df.isna().sum())

printh("h2", "List Unique Values in Each Column")
print("main_df.nunique()")
print(main_df.nunique())

printh("h2", "Has Duplicates")
print("main_df.duplicated().sum()")
print(main_df.duplicated().sum())

printh("h2", "Rearranged Columns")
LABEL = "z"
SELECT_FEATURES = [
  "x",
  "y",
]
SELECT_COLUMNS = SELECT_FEATURES + [LABEL]
main_df = main_df[SELECT_COLUMNS]

print(pd.concat([main_df[:5], main_df[-5:]]))

printh("h2", "Label")
LABEL_VALS = sorted(main_df[LABEL].unique().tolist())
print("main_df[\"%s\"].unique()" % (LABEL))
print(LABEL_VALS)

In [ ]:
# from sklearn.compose import ColumnTransformer
# from sklearn.preprocessing import OneHotEncoder
# categorical_features = ['Gender']
# preprocess = ColumnTransformer(
#     transformers=[('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)],
#     remainder='passthrough'   # keep the rest of the columns (e.g., 'sales')
# )

printh("h1", "Replace & Reformat Columns")

pd.set_option('future.no_silent_downcasting', True)
main_df = main_df.dropna()

printh("h2", "Main DF Info")
print(main_df.info())

printh("h2", "Reformatted Columns")
print(pd.concat([main_df[:5], main_df[-5:]]))

printh("h2", "Described Columns")
print(main_df.describe())

# LABEL_MAP = {idx: e for e, idx in enumerate(LABEL_VALS)}
# main_df[LABEL] = main_df[LABEL].replace(LABEL_MAP)
main_df[LABEL] = pd.to_numeric(main_df[LABEL], errors='coerce')

prep_columns = SELECT_FEATURES
for prep_col in prep_columns:
    main_df[prep_col] = pd.to_numeric(main_df[prep_col], errors='coerce')

printh("h2", "Main DF Info")
print(main_df.info())

printh("h2", "Reformatted Columns")
print(pd.concat([main_df[:5], main_df[-5:]]))

printh("h2", "Described Columns")
print(main_df.describe())

In [ ]:
printh("h1", "Correlations")
show_corr(df=main_df)

In [ ]:
printh("h1", "Histograms")

printh("h2", "Skewness + Kurtosis")
show_skew_kurt(df=main_df)

printh("h2", "Histograms")
show_hists(df=main_df, features=SELECT_COLUMNS, n=5)

In [ ]:
# printh("h1", "Scale Features & Label")

# # printh("h2", "Skewness + Kurtosis (pre-update)")
# # sk_df = show_skew_kurt(df=main_df)
# # cols = sk_df.index[sk_df["skewness"] > 1].to_list()
# # print("Fix skew features %s" % (cols))
# # for col in cols:
# #     main_df[col] = np.sqrt(main_df[col])

# # cols = ["Fare"]
# # print("Fix skew features %s" % (cols))
# # for col in cols:
# #     main_df[col] = np.cbrt(main_df[col])

# cols = [e for e in SELECT_FEATURES]
# print("Scale features %s" % (cols))
# for col in cols:
#     scaler = RobustScaler()
#     main_df[col] = scaler.fit_transform(main_df[[col]].copy())

# printh("h2", "Main DF (updated)")
# print(pd.concat([main_df[:5], main_df[-5:]]))

# printh("h2", "Skewness + Kurtosis (updated)")
# show_skew_kurt(df=main_df)

# printh("h2", "Histograms (updated)")
# show_hists(df=main_df, features=SELECT_COLUMNS, n=6)

# printh("h2", "Correlations (updated)")
# show_corr(df=main_df)

In [ ]:
printh("h1", "Split Data (train + test)")
# SELECT_FEATURES = []
# SELECT_COLUMNS = SELECT_FEATURES + [LABEL]
X = main_df[SELECT_FEATURES].to_numpy()
y = main_df[[LABEL]].to_numpy().reshape(-1)

printh("h2", "All Data")
print(X.shape)
print(X[:1])
print(y.shape)
print(y[:10])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42)

printh("h2", "Train")
print(X_train.shape)
print(X_train[:1])
print(y_train.shape)
print(y_train[:10])

printh("h2", "Test")
print(X_test.shape)
print(X_test[:1])
print(y_test.shape)
print(y_test[:10])

In [ ]:

# print(X_train.describe())
# print(y_train.describe())
# print(X_test.describe())
# print(y_test.describe())


In [ ]:
printh("h1", "Define Model")

# model = Sequential([
#     layers.Dense(30, activation='tanh', input_shape=(len(SELECT_FEATURES),)),
#     layers.Dense(20, activation='relu'),
#     layers.Dense(10, activation='relu'),
#     layers.Dense(1, activation='sigmoid'),
# ])

model = Sequential([
    layers.Dense(4, activation='tanh', input_shape=(len(SELECT_FEATURES),)),
    layers.Dense(4, activation='relu'),
    layers.Dense(1, activation='sigmoid'),
])

model.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss=losses.BinaryCrossentropy(from_logits=False),
    metrics=['accuracy']
)

In [ ]:
printh("h1", "Start Model Training")

history = model.fit(
    X_train,
    y_train,
    epochs=150,
    batch_size=40,
    validation_split=0.1,
    verbose=2
)

In [ ]:
printh("h1", "Evaluate")

loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {accuracy:.2f}")

loss, accuracy = model.evaluate(X_train, y_train, verbose=0)
print(f"Train Accuracy: {accuracy:.2f}")

In [ ]:
printh("h1", "Predict Samples")
prediction = model.predict(X_test[:10])
print(prediction)
if prediction.shape[1] > 1:
    # pred is 1-hot
    print(np.argmax(prediction, axis=1))
else:
    # pred is float 0~1
    print(np.round(prediction))

In [ ]:
printh("h1", "Create Confusion Mat")

# Make predictions
y_pred = model.predict(X_test)
if y_pred.shape[1] > 1:
    # pred is 1-hot
    y_pred = np.argmax(y_pred, axis=1)
else:
    # pred is float 0~1
    y_pred = np.round(y_pred)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy}')

# Plot the confusion matrix using Seaborn
conf_matrix = confusion_matrix(y_test, y_pred)
print(set(y_train.tolist()))
print(set(y_test.tolist()))
print(set(y_pred.flatten().tolist()))
cm_size = len(set(y_test.tolist()))
plt.figure(figsize=(cm_size, cm_size))
# sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", cbar=False, xticklabels=LABEL_VALS, yticklabels=LABEL_VALS)
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
printh("h1", "Training Info")

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy', color='blue')
plt.plot(history.history['val_accuracy'],
         label='Validation Accuracy', color='orange')
plt.title('Training and Validation Accuracy', fontsize=14)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss', color='blue')
plt.plot(history.history['val_loss'], label='Validation Loss', color='orange')
plt.title('Training and Validation Loss', fontsize=14)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend()
plt.grid(True)

plt.suptitle("Model Training Performance", fontsize=16)
plt.tight_layout()
plt.show()


In [ ]:
# printh("h1", "Feature Importances")

# def predict(X):
#     pred = model.predict(X)
#     if pred.shape[1] > 1:
#         # pred is 1-hot
#         pred = np.argmax(pred, axis=1)
#     else:
#         # pred is float 0~1
#         pred = np.round(pred)
#     return pred

# def accuracy(y_true, y_p):
#     return np.mean(y_true == y_p)

# def score(X, y):
#     y_pred = predict(X)
#     return accuracy(y, y_pred)

# printh("h2", "Get Feature Importances")
# base_score, score_decreases = get_score_importances(score, X, y)
# feature_importances = np.mean(score_decreases, axis=0)

# printh("h2", "Report")
# report = np.column_stack([SELECT_FEATURES, feature_importances])
# print(report.shape)
# print(report)
# report = np.array(sorted(report, key=lambda r: r[1], reverse=True))
# print(report.shape)
# print(report)

In [ ]:
# del(history)
# del(model)

In [ ]:
def draw_map(input_df, coor_x, coor_y, label, scale_factor=100, group_by=None, log=False):
    # lat_mid = (max(input_df[coor_x]) - min(input_df[coor_x])) / 2.0
    # lon_mid = (max(input_df[coor_y]) - min(input_df[coor_y])) / 2.0
    # input_df[coor_x] = input_df[coor_x] - (min(input_df[coor_x]) + lat_mid)
    # input_df[coor_y] = input_df[coor_y] - (min(input_df[coor_y]) + lon_mid)
    # print("Mid Point\n", input_df.describe(), "\n")
    input_df[coor_x] = np.round(input_df[coor_x] * scale_factor)
    input_df[coor_y] = np.round(input_df[coor_y] * scale_factor)
    if log:
        print("x100 round\n", input_df.describe(), input_df.shape,"\n")

    if group_by == "mean":
        input_df = input_df.groupby([coor_x,coor_y])[label].mean().reset_index()
    else:
        input_df = input_df.groupby([coor_x,coor_y])[label].first().reset_index()
    # input_df[coor_x] = input_df[coor_x] - min(input_df[coor_x])
    # input_df[coor_y] = input_df[coor_y] - min(input_df[coor_y])
    input_df[coor_x] = pd.to_numeric(input_df[coor_x], downcast="integer")
    input_df[coor_y] = pd.to_numeric(input_df[coor_y], downcast="integer")
    if log:
        print("GroupByXYAveragePrice\n", input_df.describe(), input_df.shape, "\n")
    
    min_coor_x = min(input_df[coor_x])
    max_coor_x = max(input_df[coor_x])
    min_coor_y = min(input_df[coor_y])
    max_coor_y = max(input_df[coor_y])
    width = int(max(input_df[coor_x]) - min(input_df[coor_x]))
    length = int(max(input_df[coor_y]) - min(input_df[coor_y]))
    Z = np.full((length, width), -1)
    x = np.arange(min_coor_x, max_coor_x, 1) / scale_factor
    y = np.arange(min_coor_y, max_coor_y, 1) / scale_factor
    if log:
        # print(x, y)
        print("width", "length", width, length)
        printh("h2", "Reformatted Columns")
        print(pd.concat([input_df[:5], input_df[-5:]]))
        printh("h2", "Described Columns")
        print(input_df.describe())
        printh("h2", "Plot")
        print(Z.shape)
    for idx in range(0, width):
        for jdx in range(0, length):
            value = input_df.loc[(input_df[coor_x] == min_coor_x+idx) & (input_df[coor_y] == min_coor_y+jdx), label]
            if len(value) > 0:
                Z[jdx][idx] = value.iloc[0]

    fig, ax = plt.subplots(figsize=(10, 10))
    ax.set_title(label)
    ax.set_xlabel(coor_x)
    ax.set_ylabel(coor_y)
    # ax.set_box_aspect(1.0)
    mesh = ax.pcolormesh(x, y, Z, cmap="viridis")
    fig.colorbar(mesh, ax=ax)
    plt.show()




infer_df = pd.read_csv('noisy_.csv')
infer_df = infer_df[SELECT_FEATURES]
print(infer_df.head())

prediction = model.predict(infer_df)
# print(prediction[:20])
if prediction.shape[1] > 1:
    # pred is 1-hot
    prediction = np.argmax(prediction, axis=1)
else:
    # pred is float 0~1
    prediction = np.round(prediction)
# print(prediction[:20])
infer_df[LABEL] = prediction
print(infer_df.head())


draw_map(
    input_df=infer_df,
    coor_x="x",
    coor_y="y",
    label=LABEL,
    scale_factor=100,
)

